In [ ]:
import os
import pandas as pd
import mlflow
import mlflow.sklearn
import mlflow.pyfunc
from dotenv import load_dotenv
from sqlalchemy import create_engine
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline 

In [2]:
# подгружаем .env
load_dotenv()

True

In [3]:
# Считываем все креды
src_host = os.environ.get('DB_SOURCE_HOST')
src_port = os.environ.get('DB_SOURCE_PORT')
src_username = os.environ.get('DB_SOURCE_USER')
src_password = os.environ.get('DB_SOURCE_PASSWORD')
src_db = os.environ.get('DB_SOURCE_NAME') 

dst_host = os.environ.get('DB_DESTINATION_HOST')
dst_port = os.environ.get('DB_DESTINATION_PORT')
dst_username = os.environ.get('DB_DESTINATION_USER')
dst_password = os.environ.get('DB_DESTINATION_PASSWORD')
dst_db = os.environ.get('DB_DESTINATION_NAME')

s3_bucket = os.environ.get('S3_BUCKET_NAME')
s3_access_key = os.environ.get('AWS_ACCESS_KEY_ID')
s3_secret_access_key = os.environ.get('AWS_SECRET_ACCESS_KEY')

In [4]:
# Создадим соединения
src_conn = create_engine(f'postgresql://{src_username}:{src_password}@{src_host}:{src_port}/{src_db}')
dst_conn = create_engine(f'postgresql://{dst_username}:{dst_password}@{dst_host}:{dst_port}/{dst_db}')

In [5]:
# Укажем таблицу для удаления
TABLE_NAME = 'clean_users_churn'

In [6]:
df = pd.read_sql(f'select * from {TABLE_NAME}', dst_conn)

In [7]:
src_conn.dispose()
dst_conn.dispose()

In [8]:
df.head()

,id,customer_id,begin_date,end_date,type,paperless_billing,payment_method,monthly_charges,total_charges,internet_service,...,device_protection,tech_support,streaming_tv,streaming_movies,gender,senior_citizen,partner,dependents,multiple_lines,target
0,2133,3023-GFLBR,2017-03-01,2019-12-01,Month-to-month,No,Credit card (automatic),86.15,2745.70,Fiber optic,...,No,No,No,Yes,Female,0,Yes,Yes,Yes,1
1,837,0727-BMPLR,2015-04-01,2019-11-01,One year,Yes,Electronic check,100.00,5509.30,Fiber optic,...,Yes,No,Yes,Yes,Female,1,No,No,Yes,1
2,890,9227-LUNBG,2019-10-01,2019-11-01,Month-to-month,No,Electronic check,24.60,24.60,DSL,...,No,No,No,No,Female,0,No,No,No,1
3,1001,7047-YXDMZ,2018-05-01,2019-11-01,Month-to-month,No,Mailed check,20.00,417.70,Fiber optic,...,No,No,No,No,Male,0,No,No,No,0
4,1002,2858-EIMXH,2015-09-01,2019-11-01,One year,Yes,Credit card (automatic),95.85,5016.25,Fiber optic,...,No,No,Yes,Yes,Female,1,Yes,No,Yes,0


In [9]:
df.drop(['id', 'customer_id'], axis=1, inplace=True)

In [10]:
# Разделите данные на тренировочную и тестовую выборки
X = df.drop(columns=['target'])
y = df['target']
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)

# Определите категориальные и числовые признаки
cat_features = X_train.select_dtypes(include='object')
num_features = X_train.select_dtypes(include=['float'])

# Создайте трансформеры и модель
preprocessor = ColumnTransformer(
    [
        ('cat', OneHotEncoder(drop='if_binary'), cat_features.columns.tolist()),
        ('num', StandardScaler(), num_features.columns.tolist())
    ],
    remainder='drop',
    verbose_feature_names_out=False
)

model = LogisticRegression(
    C=1, 
    penalty='l2'
)

# Объедините всё в пайплайн
pipeline = Pipeline(
    [
        ('preprocessor', preprocessor),
        ('model', model)
    ]
)

# Обучите модель на тренировочной выборке
pipeline.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('cat',
                                                  OneHotEncoder(drop='if_binary'),
                                                  ['type', 'paperless_billing',
                                                   'payment_method',
                                                   'internet_service',
                                                   'online_security',
                                                   'online_backup',
                                                   'device_protection',
                                                   'tech_support',
                                                   'streaming_tv',
                                                   'streaming_movies', 'gender',
                                                   'partner', 'dependents',
                                                   'multiple_lines']),
                                                 ('num', StandardScaler(),
                                                  ['monthly_charges',
                                                   'total_charges'])],
                                   verbose_feature_names_out=False)),
                ('model', LogisticRegression(C=1))])

In [11]:
from sklearn.metrics import confusion_matrix, roc_auc_score, precision_score, recall_score, f1_score, log_loss

# Предсказанные значения и вероятности
proba = pipeline.predict_proba(X_test)[:, 1]
prediction = pipeline.predict(X_test)

# Истинные метки и данные для предсказания
y_true = y_test

# Заведите словарь со всеми метриками
metrics = {}

# Посчитайте метрики из модуля sklearn.metrics с нормализацией
_, err1, _, err2 = confusion_matrix(y_test, prediction, normalize='all').ravel()

auc = roc_auc_score(y_true, proba)
precision = precision_score(y_true, prediction)
recall = recall_score(y_true, prediction)
f1 = f1_score(y_true, prediction)
logloss = log_loss(y_true, proba)

# Запишите значения метрик в словарь 
metrics["err1"] = err1
metrics["err2"] = err2
metrics["auc"] = auc
metrics["precision"] = precision
metrics["recall"] = recall
metrics["f1"] = f1
metrics["logloss"] = logloss

In [12]:
model = pipeline.named_steps['model']

In [13]:
os.environ["MLFLOW_S3_ENDPOINT_URL"] = 'https://storage.yandexcloud.net' # ваш код здесь
os.environ["AWS_ACCESS_KEY_ID"] = os.getenv('AWS_ACCESS_KEY_ID') # ваш код здесь
os.environ["AWS_SECRET_ACCESS_KEY"] = os.getenv('AWS_SECRET_ACCESS_KEY') # ваш код здесь 

In [14]:
TRACKING_SERVER_HOST = "127.0.0.1"
TRACKING_SERVER_PORT = 5000

# напишите код, который подключает tracking и registry uri
mlflow.set_tracking_uri(f'http://{TRACKING_SERVER_HOST}:{TRACKING_SERVER_PORT}')
mlflow.set_registry_uri(f'http://{TRACKING_SERVER_HOST}:{TRACKING_SERVER_PORT}')

In [16]:
EXPERIMENT_NAME = 'churn_laptev_ilya_sergeevich_2'
RUN_NAME = "model_1_registry"
REGISTRY_MODEL_NAME = "churn_model_laptev_ilya_sergeevich_2"

pip_requirements = '../requirements.txt'
signature = mlflow.models.infer_signature(X_test, prediction)
input_example = X_test[:10]
metadata = {'model_type': 'monthly'}

experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)
if experiment is None:
    experiment_id = mlflow.create_experiment(EXPERIMENT_NAME)
else:
    experiment_id = experiment.experiment_id

with mlflow.start_run(run_name=RUN_NAME, experiment_id=experiment_id) as run:
    run_id = run.info.run_id

    mlflow.log_metrics(metrics)
    # ваш код здесь
    model_info = mlflow.sklearn.log_model(sk_model=model,
                                          metadata=metadata,
                                          artifact_path='models',
                                          signature=signature,
                                          pip_requirements=pip_requirements,
                                          input_example=input_example,
                                          registered_model_name=REGISTRY_MODEL_NAME,
                                          await_registration_for=60)


/home/mle-user/mle-mlflow/.venv_mle_mlflow/lib/python3.10/site-packages/mlflow/models/signature.py:212: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  inputs = _infer_schema(model_input) if model_input is not None else None
Registered model 'churn_model_laptev_ilya_sergeevich_2' already exists. Creating a new version of this model...
2024/11/02 12:26:50 INFO mlflow.tracking.

In [17]:
client = mlflow.MlflowClient()


models = client.search_model_versions(filter_string=f"name = '{REGISTRY_MODEL_NAME}'")
print(f"Model info:\n {models}")

Model info:
 [<ModelVersion: aliases=[], creation_timestamp=1730550410205, current_stage='None', description='', last_updated_timestamp=1730550410205, name='churn_model_laptev_ilya_sergeevich_2', run_id='a9d570e6477043b6bef9a780876ac4ca', run_link='', source='s3://s3-student-mle-20240920-1460ff9140/4/a9d570e6477043b6bef9a780876ac4ca/artifacts/models', status='READY', status_message='', tags={}, user_id='', version='3'>, <ModelVersion: aliases=[], creation_timestamp=1730541195254, current_stage='Production', description='', last_updated_timestamp=1730541285515, name='churn_model_laptev_ilya_sergeevich_2', run_id='eaa9b68d72e344a78aea5a99d08d677e', run_link='', source='s3://s3-student-mle-20240920-1460ff9140/4/eaa9b68d72e344a78aea5a99d08d677e/artifacts/models', status='READY', status_message='', tags={}, user_id='', version='2'>, <ModelVersion: aliases=[], creation_timestamp=1730541173843, current_stage='Staging', description='', last_updated_timestamp=1730541256589, name='churn_model_la

In [22]:
model_name_1 = models[-1].name
model_version_1 = models[-1].version
model_stage_1 = models[-1].current_stage

model_name_2 = models[-2].name
model_version_2 = models[-2].version
model_stage_2 = models[-2].current_stage

print(f"Текущий stage модели 1: {model_stage_1}")
print(f"Текущий stage модели 2: {model_stage_2}")

Текущий stage модели 1: Staging
Текущий stage модели 2: Production


In [23]:
# поменяйте статус каждой модели
client.transition_model_version_stage(model_name_1, model_version_1, "production")
client.transition_model_version_stage(model_name_2, model_version_2, "staging")

<ModelVersion: aliases=[], creation_timestamp=1730541195254, current_stage='Staging', description='', last_updated_timestamp=1730550986437, name='churn_model_laptev_ilya_sergeevich_2', run_id='eaa9b68d72e344a78aea5a99d08d677e', run_link='', source='s3://s3-student-mle-20240920-1460ff9140/4/eaa9b68d72e344a78aea5a99d08d677e/artifacts/models', status='READY', status_message='', tags={}, user_id='', version='2'>

In [24]:
# Переименовываем модель в реестре, добавляя _b2c
client.rename_registered_model(name=REGISTRY_MODEL_NAME, new_name=f'{REGISTRY_MODEL_NAME}_b2c')

In [15]:
# run.info.run_id
model_info.model_uri

'runs:/d98aa0f616124f78a8a443f1418e04b8/models'

In [74]:
loaded_model = mlflow.pyfunc.load_model(model_info.model_uri)

In [76]:
# Преобразуйте категориальные признаки
cat_transformer = OneHotEncoder(drop='if_binary', sparse=False)
cat_transformer.fit(X_train[cat_features.columns])
X_train_cat = cat_transformer.transform(X_train[cat_features.columns])
X_test_cat = cat_transformer.transform(X_test[cat_features.columns])

# Преобразуйте их в DataFrame для удобства конкатенации
X_train_cat_df = pd.DataFrame(X_train_cat, index=X_train.index, columns=cat_transformer.get_feature_names_out())
X_test_cat_df = pd.DataFrame(X_test_cat, index=X_test.index, columns=cat_transformer.get_feature_names_out())

# Преобразуйте числовые признаки
num_transformer = StandardScaler()
num_transformer.fit(X_train[num_features.columns])
X_train_num = num_transformer.transform(X_train[num_features.columns])
X_test_num = num_transformer.transform(X_test[num_features.columns])

# Преобразуйте их в DataFrame для удобства конкатенации
X_train_num_df = pd.DataFrame(X_train_num, index=X_train.index, columns=num_features.columns)
X_test_num_df = pd.DataFrame(X_test_num, index=X_test.index, columns=num_features.columns)

# Объедините преобразованные признаки с использованием concat
X_train_processed = pd.concat([X_train_num_df, X_train_cat_df], axis=1)
X_test_processed = pd.concat([X_test_num_df, X_test_cat_df], axis=1)

/home/mle-user/mle-mlflow/.venv_mle_mlflow/lib/python3.10/site-packages/sklearn/preprocessing/_encoders.py:975: FutureWarning: `sparse` was renamed to `sparse_output` in version 1.2 and will be removed in 1.4. `sparse_output` is ignored unless you leave `sparse` to its default value.
  warnings.warn(
